# Load Embeddings

In [ ]:
import os
# change working directory so that it picks up the grapehelper library
os.chdir('/home/ftorgano/rna-kg-analysis')
print(os.getcwd())

In [ ]:
from helper_lib import graph
from helper_lib import cache
from helper_lib import predict
import importlib
import logging

importlib.reload(graph)
importlib.reload(cache)
importlib.reload(predict)
cache.set_embedding_cache_dir("./RNA-KG_notebooks/Default_RNA-KG/cache/embeddings/")

logging.basicConfig(level=logging.INFO)
logging.getLogger().setLevel(logging.INFO)

In [ ]:
import pandas as pd
from itables import init_notebook_mode
import time
import numpy as np
import matplotlib.pyplot as plt
from cycler import cycler

init_notebook_mode(all_interactive=False)
cycler_colors = ["#3f90da", "#ffa90e", "#bd1f01", "#94a4a2", "#832db6", "#a96b59", "#e76300", "#b9ac70", "#717581", "#92dadd"]

plt.rcParams['axes.prop_cycle'] = cycler(color=cycler_colors)
df_formatters = {'balanced_acc_mean':'{:.2%}'.format,'balanced_acc_std':'{:.2%}'.format}

In [ ]:
undirected_rnakg = graph.load_rnakg_fixed()

In [ ]:
len(undirected_rnakg.get_node_type_names_counts_hashmap())

In [ ]:
len(undirected_rnakg.get_edge_type_names_counts_hashmap())

In [ ]:
from grape.embedders import Node2VecSkipGramEnsmallen
# try to load the embedding
try:
    n2v_sg_bfs_100 = cache.load_embeddings('Node2VecSkipGramEnsmallen_BFS_100_fixed')
    print(type(n2v_sg_bfs_100))
except:
    # creating smaller embeddings - 100 dimensions
    start_time = time.time()
    embedding_n2v_sp_bfs_res_100 = Node2VecSkipGramEnsmallen(return_weight=5, explore_weight=0.2,embedding_size=100)\
        .fit_transform(undirected_rnakg)
    cache.cache_embedding(embedding_n2v_sp_bfs_res_100, 'Node2VecSkipGramEnsmallen_BFS_100_fixed')
    n2v_sg_bfs_100 = cache.load_embeddings('Node2VecSkipGramEnsmallen_BFS_100_fixed')
    end_time = time.time()
embedding_100 = n2v_sg_bfs_100[0]

In [ ]:
from grape.embedders import Node2VecSkipGramEnsmallen
# try to load the embedding
try:
    n2v_sg_bfs_10 = cache.load_embeddings('Node2VecSkipGramEnsmallen_BFS_10_fixed')
    print(type(n2v_sg_bfs_10))
except:
    # creating smaller embeddings - 10 dimensions
    start_time = time.time()
    embedding_n2v_sp_bfs_res_10 = Node2VecSkipGramEnsmallen(return_weight=5, explore_weight=0.2,embedding_size=10)\
        .fit_transform(undirected_rnakg)
    cache.cache_embedding(embedding_n2v_sp_bfs_res_10, 'Node2VecSkipGramEnsmallen_BFS_10_fixed')
    n2v_sg_bfs_10 = cache.load_embeddings('Node2VecSkipGramEnsmallen_BFS_10_fixed')
    end_time = time.time()
embedding_10 = n2v_sg_bfs_10[0]

In [ ]:
from grape.embedders import Node2VecSkipGramEnsmallen
# try to load the embedding
try:
    n2v_sg_bfs_50 = cache.load_embeddings('Node2VecSkipGramEnsmallen_BFS_50_fixed')
    print(type(n2v_sg_bfs_50))
except:
    # creating smaller embeddings - 50 dimensions
    print("Creating 50 dimensions embedding")
    start_time = time.time()
    embedding_n2v_sp_bfs_res_50 = Node2VecSkipGramEnsmallen(return_weight=5, explore_weight=0.2,embedding_size=50)\
        .fit_transform(undirected_rnakg)
    cache.cache_embedding(embedding_n2v_sp_bfs_res_50, 'Node2VecSkipGramEnsmallen_BFS_50_fixed')
    n2v_sg_bfs_50 = cache.load_embeddings('Node2VecSkipGramEnsmallen_BFS_50_fixed')
    end_time = time.time()
embedding_50 = n2v_sg_bfs_50[0]

In [ ]:
def classes_to_remove_top_k(embedding_types, k):
    unique, counts = np.unique(embedding_types, return_counts=True)
    class_count = sorted(zip(unique, counts), key=lambda x: x[1], reverse=True)
    return [x[0] for x in class_count[k:]]
def set_classes_as_removed(embedding_types, classes_to_remove):
    embedding_types_filtered = embedding_types.copy().astype(int)
    # label_removed_class = np.max(embedding_types) + 1
    for i in range(len(embedding_types)):
        if embedding_types[i] in classes_to_remove:
            embedding_types_filtered[i] = -1
    return embedding_types_filtered

In [ ]:
def get_flatten_multi_label_and_unknown_node_types(graph) -> np.ndarray:
        # BORROWED GRAPE FUNCTION
        """Returns flattened node type IDs adjusted for the current instance."""
        # The following is needed to normalize the multiple types
        node_types_counts = graph.get_node_type_id_counts_hashmap()
        top_10_node_types = {
            node_type: 50 - i
            for i, node_type in enumerate(
                sorted(node_types_counts.items(), key=lambda x: x[1], reverse=True)[:50]
            )
        }
        node_types_counts = {
            node_type: top_10_node_types.get(node_type, 0)
            for node_type in node_types_counts
        }
        node_types_number = graph.get_number_of_node_types()
        unknown_node_types_id = node_types_number

        # When we have multiple node types for a given node, we set it to
        # the most common node type of the set.
        return np.fromiter(
            (
                unknown_node_types_id
                if node_type_ids is None
                else sorted(
                    node_type_ids,
                    key=lambda node_type: node_types_counts[node_type],
                    reverse=True,
                )[0]
                for node_type_ids in (
                    graph.get_node_type_ids_from_node_id(node_id)
                    for node_id in range(graph.get_number_of_nodes())
                )
            ),
            dtype=np.uint32,
        )

In [ ]:
embedding_types = get_flatten_multi_label_and_unknown_node_types(undirected_rnakg)

In [ ]:
embedding_types

## Cleaning dataset

In [ ]:
unique, counts = np.unique(embedding_types, return_counts=True)
class_count = dict(zip(unique, counts))

classes_to_remove = []
for (k,v) in class_count.items():
    if v < 2:
        classes_to_remove.append(k)

In [ ]:
indices_to_remove = []
for i,c in enumerate(embedding_types):
    if c in classes_to_remove:
        indices_to_remove.append(i)

In [ ]:
embedding_10_filtered = np.delete(embedding_10, indices_to_remove, axis=0)
embedding_50_filtered = np.delete(embedding_50, indices_to_remove, axis=0)
embedding_100_filtered = np.delete(embedding_100, indices_to_remove, axis=0)
embedding_types_filtered = np.delete(embedding_types, indices_to_remove, axis=0)

# Prediction

In [ ]:
setup = {
    'keep_top_classes_all': [7, 20, 54, 68, 81],
    'n_splits': 5,
    'test_size': 0.3,
    'random_state': 42,
}

## Tree Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier

### Node type prediction - 100 dim

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

train_test_sets_100 = []

split = StratifiedShuffleSplit(n_splits=setup['n_splits'], test_size=setup['test_size'], random_state=setup['random_state']) 
for train_index, test_index in split.split(embedding_100_filtered, embedding_types_filtered):
    train_test_set = {
        'train': (embedding_100_filtered[train_index], embedding_types_filtered[train_index]),
        'test': (embedding_100_filtered[test_index], embedding_types_filtered[test_index])
    }
    train_test_sets_100.append(train_test_set)

#### Tuning

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

param_grid = {
    'tree__max_depth': [5, 10, 15, 20, 25, 35, 40, 45, 50, 60, 70, 80],
    'tree__max_features': [5,'sqrt',40,60,80,100],
}

for train_test_set in train_test_sets_100:
    pipe = Pipeline(steps=[('scaler', StandardScaler()), ('tree', DecisionTreeClassifier(random_state=setup['random_state']))])
    grid_100 = GridSearchCV(pipe, param_grid=param_grid, n_jobs=1, scoring='balanced_accuracy')
    grid_100.fit(train_test_set['train'][0], train_test_set['train'][1])
    result_grid_100 = pd.concat([pd.DataFrame(grid_100.cv_results_["params"]),pd.DataFrame(grid_100.cv_results_["mean_test_score"], columns=["Accuracy"]),pd.DataFrame(grid_100.cv_results_["mean_fit_time"],columns=["fit_time"])],axis=1).sort_values(by=["Accuracy"],ascending=False)
    print(result_grid_100.head(5))

#### Best model

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import balanced_accuracy_score

fixed_parameters = {'random_state':setup['random_state'], 'max_features':100, 'max_depth':40}

classes_to_remove_top = []
for keep_top_classes in setup['keep_top_classes_all']:
    classes = classes_to_remove_top_k(embedding_types=embedding_types, k=keep_top_classes)
    classes_to_remove_top.append(classes)

final_results_100 = {}

for elem in setup['keep_top_classes_all']:
    final_results_100[elem] = {'train':[], 'test':[]}

for i,train_test_set in enumerate(train_test_sets_100): 
    print(f'Holdout {i}/{setup["n_splits"]}')
    for i,keep_top_classes in enumerate(setup['keep_top_classes_all']): ########################
        print(f'classes: {keep_top_classes}')
        train_set = train_test_set['train']
        x_train = train_set[0]
        y_train = train_set[1]
        test_set = train_test_set['test']
        x_test = test_set[0]
        y_test = test_set[1]

        y_train_filtered = set_classes_as_removed(y_train, classes_to_remove_top[i])
        y_test_filtered = set_classes_as_removed(y_test, classes_to_remove_top[i])

        pipe = Pipeline(steps=[('scaler', StandardScaler()), ('tree', DecisionTreeClassifier(**fixed_parameters))])

        pipe.fit(x_train, y_train_filtered)
        
        train_score = balanced_accuracy_score(y_train_filtered, pipe.predict(x_train))
        test_score = balanced_accuracy_score(y_test_filtered, pipe.predict(x_test))
        print(f'train balanced accuracy: {train_score}')
        print(f'test balanced accuracy: {test_score}')
        final_results_100[keep_top_classes]['train'].append(train_score)
        final_results_100[keep_top_classes]['test'].append(test_score)


In [ ]:
rows = []

for keep_top_classes in setup['keep_top_classes_all']:
    for i in range(setup['n_splits']):
        rows.append({
            'holdout': i, 
            'keep_top_classes': keep_top_classes,
            'train_balanced_acc_mean': final_results_100[keep_top_classes]['train'][i],
            'test_balanced_acc_mean': final_results_100[keep_top_classes]['test'][i]
        })

df_100 = pd.DataFrame(rows)
df_100.to_csv('FullDim_NTP_DecisionTreeClassifier_100.csv', index=False)

### Node type prediction - 10 dim

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

train_test_sets_10 = []

split = StratifiedShuffleSplit(n_splits=setup['n_splits'], test_size=setup['test_size'], random_state=setup['random_state']) 
for train_index, test_index in split.split(embedding_10_filtered, embedding_types_filtered):
    train_test_set = {
        'train': (embedding_10_filtered[train_index], embedding_types_filtered[train_index]),
        'test': (embedding_10_filtered[test_index], embedding_types_filtered[test_index])
    }
    train_test_sets_10.append(train_test_set)

#### Tuning

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler


param_grid = {
    'tree__max_depth': [5,10,15,20,25,40,50,60,70,80], 
    'tree__max_features': [1,'sqrt',5,7,9,10], 
}

for train_test_set in train_test_sets_10:
    pipe = Pipeline(steps=[('scaler', StandardScaler()), ('tree', DecisionTreeClassifier(random_state=setup['random_state']))])
    grid_10 = GridSearchCV(pipe, param_grid=param_grid, n_jobs=1, scoring='balanced_accuracy')
    grid_10.fit(train_test_set['train'][0], train_test_set['train'][1])
    result_grid_10 = pd.concat([pd.DataFrame(grid_10.cv_results_["params"]),pd.DataFrame(grid_10.cv_results_["mean_test_score"], columns=["Accuracy"]),pd.DataFrame(grid_10.cv_results_["mean_fit_time"],columns=["fit_time"])],axis=1).sort_values(by=["Accuracy"],ascending=False)
    print(result_grid_10.head(5))

#### Best model

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import balanced_accuracy_score

fixed_parameters = {'random_state':setup['random_state'], 'max_features':5, 'max_depth':50}

classes_to_remove_top = []
for keep_top_classes in setup['keep_top_classes_all']:
    classes = classes_to_remove_top_k(embedding_types=embedding_types, k=keep_top_classes)
    classes_to_remove_top.append(classes)

final_results_10 = {}

for elem in setup['keep_top_classes_all']:
    final_results_10[elem] = {'train':[], 'test':[]}

for train_test_set in train_test_sets_10: 
    for i,keep_top_classes in enumerate(setup['keep_top_classes_all']): ########################
        print(f'classes: {keep_top_classes}')
        train_set = train_test_set['train']
        x_train = train_set[0]
        y_train = train_set[1]
        test_set = train_test_set['test']
        x_test = test_set[0]
        y_test = test_set[1]

        y_train_filtered = set_classes_as_removed(y_train, classes_to_remove_top[i])
        y_test_filtered = set_classes_as_removed(y_test, classes_to_remove_top[i])

        pipe = Pipeline(steps=[('scaler', StandardScaler()), ('tree', DecisionTreeClassifier(**fixed_parameters))])

        pipe.fit(x_train, y_train_filtered)
        
        train_score = balanced_accuracy_score(y_train_filtered, pipe.predict(x_train))
        test_score = balanced_accuracy_score(y_test_filtered, pipe.predict(x_test))
        print(f'train balanced accuracy: {train_score}')
        print(f'test balanced accuracy: {test_score}')
        final_results_10[keep_top_classes]['train'].append(train_score)
        final_results_10[keep_top_classes]['test'].append(test_score)


In [ ]:
rows = []

for keep_top_classes in setup['keep_top_classes_all']:
    for i in range(setup['n_splits']):
        rows.append({
            'holdout': i, 
            'keep_top_classes': keep_top_classes,
            'train_balanced_acc_mean': final_results_10[keep_top_classes]['train'][i],
            'test_balanced_acc_mean': final_results_10[keep_top_classes]['test'][i]
        })

df_10 = pd.DataFrame(rows)
df_10.to_csv('FullDim_NTP_DecisionTreeClassifier_10.csv', index=False)

### Node type prediction - 50 dim

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

train_test_sets_50 = []

split = StratifiedShuffleSplit(n_splits=setup['n_splits'], test_size=setup['test_size'], random_state=setup['random_state']) 
for train_index, test_index in split.split(embedding_50_filtered, embedding_types_filtered):
    train_test_set = {
        'train': (embedding_50_filtered[train_index], embedding_types_filtered[train_index]),
        'test': (embedding_50_filtered[test_index], embedding_types_filtered[test_index])
    }
    train_test_sets_50.append(train_test_set)

#### Tuning

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler


param_grid = {
    'tree__max_depth': [25,30,40,50], 
    'tree__max_features': [5,'sqrt',10,20,35,50], 
}

for train_test_set in train_test_sets_50:
    pipe = Pipeline(steps=[('scaler', StandardScaler()), ('tree', DecisionTreeClassifier(random_state=setup['random_state']))])
    grid_50 = GridSearchCV(pipe, param_grid=param_grid, n_jobs=1, scoring='balanced_accuracy')
    grid_50.fit(train_test_set['train'][0], train_test_set['train'][1])
    result_grid_50 = pd.concat([pd.DataFrame(grid_50.cv_results_["params"]),pd.DataFrame(grid_50.cv_results_["mean_test_score"], columns=["Accuracy"]),pd.DataFrame(grid_50.cv_results_["mean_fit_time"],columns=["fit_time"])],axis=1).sort_values(by=["Accuracy"],ascending=False)
    print(result_grid_50.head(5))

#### Best model

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import balanced_accuracy_score

fixed_parameters = {'random_state':setup['random_state'], 'max_features':20, 'max_depth':40}

classes_to_remove_top = []
for keep_top_classes in setup['keep_top_classes_all']:
    classes = classes_to_remove_top_k(embedding_types=embedding_types, k=keep_top_classes)
    classes_to_remove_top.append(classes)

final_results_50 = {}

for elem in setup['keep_top_classes_all']:
    final_results_50[elem] = {'train':[], 'test':[]}

for train_test_set in train_test_sets_50: 
    for i,keep_top_classes in enumerate(setup['keep_top_classes_all']): ########################
        print(f'classes: {keep_top_classes}')
        train_set = train_test_set['train']
        x_train = train_set[0]
        y_train = train_set[1]
        test_set = train_test_set['test']
        x_test = test_set[0]
        y_test = test_set[1]

        y_train_filtered = set_classes_as_removed(y_train, classes_to_remove_top[i])
        y_test_filtered = set_classes_as_removed(y_test, classes_to_remove_top[i])

        pipe = Pipeline(steps=[('scaler', StandardScaler()), ('tree', DecisionTreeClassifier(**fixed_parameters))])

        pipe.fit(x_train, y_train_filtered)
        
        train_score = balanced_accuracy_score(y_train_filtered, pipe.predict(x_train))
        test_score = balanced_accuracy_score(y_test_filtered, pipe.predict(x_test))
        print(f'train balanced accuracy: {train_score}')
        print(f'test balanced accuracy: {test_score}')
        final_results_50[keep_top_classes]['train'].append(train_score)
        final_results_50[keep_top_classes]['test'].append(test_score)


In [ ]:
rows = []

for keep_top_classes in setup['keep_top_classes_all']:
    for i in range(setup['n_splits']):
        rows.append({
            'holdout': i, 
            'keep_top_classes': keep_top_classes,
            'train_balanced_acc_mean': final_results_50[keep_top_classes]['train'][i],
            'test_balanced_acc_mean': final_results_50[keep_top_classes]['test'][i]
        })

df_50 = pd.DataFrame(rows)
df_50.to_csv('FullDim_NTP_DecisionTreeClassifier_50.csv', index=False)

## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

### Node type prediction - 10 dim

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

train_test_sets_10 = []

split = StratifiedShuffleSplit(n_splits=setup['n_splits'], test_size=setup['test_size'], random_state=setup['random_state']) 
for train_index, test_index in split.split(embedding_10_filtered, embedding_types_filtered):
    train_test_set = {
        'train': (embedding_10_filtered[train_index], embedding_types_filtered[train_index]),
        'test': (embedding_10_filtered[test_index], embedding_types_filtered[test_index])
    }
    train_test_sets_10.append(train_test_set)

#### Tuning

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

param_grid = {
    'forest__max_depth': [50,100,150,200,250,300], 
    'forest__max_features': [3,5,7,10], 
}

for train_test_set in train_test_sets_10:
    pipe = Pipeline(steps=[('scaler', StandardScaler()), ('forest', RandomForestClassifier(random_state=setup['random_state'],n_jobs=-1))])
    grid_10 = GridSearchCV(pipe, param_grid=param_grid, n_jobs=1, scoring='balanced_accuracy')
    grid_10.fit(train_test_set['train'][0], train_test_set['train'][1])
    result_grid_10 = pd.concat([pd.DataFrame(grid_10.cv_results_["params"]),pd.DataFrame(grid_10.cv_results_["mean_test_score"], columns=["Accuracy"]),pd.DataFrame(grid_10.cv_results_["mean_fit_time"],columns=["fit_time"])],axis=1).sort_values(by=["Accuracy"],ascending=False)
    print(result_grid_10.head(5))

#### Best model

In [ ]:
from sklearn.metrics import balanced_accuracy_score

fixed_parameters = {'random_state':setup['random_state'],'n_jobs':-1, 'n_estimators':300, 'max_features':5}

classes_to_remove_top = []
for keep_top_classes in setup['keep_top_classes_all']:
    classes = classes_to_remove_top_k(embedding_types=embedding_types, k=keep_top_classes)
    classes_to_remove_top.append(classes)

final_results_10_forest = {}

for elem in setup['keep_top_classes_all']:
    final_results_10_forest[elem] = {'train':[], 'test':[]}

for train_test_set in train_test_sets_10: 
    for i,keep_top_classes in enumerate(setup['keep_top_classes_all']): ########################
        print(f'classes: {keep_top_classes}')
        train_set = train_test_set['train']
        x_train = train_set[0]
        y_train = train_set[1]
        test_set = train_test_set['test']
        x_test = test_set[0]
        y_test = test_set[1]

        y_train_filtered = set_classes_as_removed(y_train, classes_to_remove_top[i])
        y_test_filtered = set_classes_as_removed(y_test, classes_to_remove_top[i])

        pipe = Pipeline(steps=[('scaler', StandardScaler()), ('forest', RandomForestClassifier(**fixed_parameters))])

        pipe.fit(x_train, y_train_filtered)
        
        train_score = balanced_accuracy_score(y_train_filtered, pipe.predict(x_train))
        test_score = balanced_accuracy_score(y_test_filtered, pipe.predict(x_test))
        print(f'train balanced accuracy: {train_score}')
        print(f'test balanced accuracy: {test_score}')
        final_results_10_forest[keep_top_classes]['train'].append(train_score)
        final_results_10_forest[keep_top_classes]['test'].append(test_score)

In [ ]:
rows = []

for keep_top_classes in setup['keep_top_classes_all']:
    for i in range(setup['n_splits']):
        rows.append({
            'holdout': i, 
            'keep_top_classes': keep_top_classes,
            'train_balanced_acc_mean': final_results_10_forest[keep_top_classes]['train'][i],
            'test_balanced_acc_mean': final_results_10_forest[keep_top_classes]['test'][i]
        })

df_10_forest = pd.DataFrame(rows)
df_10_forest.to_csv('FullDim_NTP_RandomForestClassifier_10.csv', index=False)